### Deflection of a plate using the CLPT

In [18]:
!python -m pip install buckling structsolve plotly > tmp.txt

In [19]:
import numpy as np
from scipy.sparse import csc_matrix
from scipy.special import roots_legendre
from composites import isotropic_plate
from structsolve import solve

from buckling.legendre import vecf, vecfxi, vecfxixi

# approximation order
m1 = m2 = 20
N = m1*m2
print('m1 = %d' % m1)
print('m2 = %d' % m2)
print('N = %d' % N)
i = np.arange(m1)
j = np.arange(m2)
pts1, weights1 = roots_legendre(2*m1 - 1)
pts2, weights2 = roots_legendre(2*m2 - 1)

# Material properties
E = 200.e9
nu = 0.3
G = E/(2*(1 + nu))

# Boundary conditions (simply supported)
xit1 = 0
xir1 = 0
xit2 = 0
xir2 = 0
etat1 = 0
etar1 = 0
etat2 = 0
etar2 = 0

# Geometric properties
a = 3
b = 7
h = 0.005

# Calculating ABD matrices
prop = isotropic_plate(thickness=h, E=E, nu=nu)

buff = np.zeros((N, N))
K = np.zeros((N, N))

def addouter(matrix, vec1, vec2):
    np.outer(vec1, vec2, out=buff)
    matrix += buff

# stiffness matrix
# numerical integration in 2D using Legendre-Gauss quadrature
for xi, wxi in zip(pts1, weights1):
    P_xi = vecf(m1, xi, xit1, xir1, xit2, xir2)
    Px_xi = vecfxi(m1, xi, xit1, xir1, xit2, xir2)
    Pxx_xi = vecfxixi(m1, xi, xit1, xir1, xit2, xir2)
    for eta, weta in zip(pts2, weights2):
        P_eta = vecf(m2, eta, etat1, etar1, etat2, etar2)
        Px_eta = vecfxi(m2, eta, etat1, etar1, etat2, etar2)
        Pxx_eta = vecfxixi(m2, eta, etat1, etar1, etat2, etar2)

        weight = wxi*weta

        Pi, Pj = np.meshgrid(P_xi, P_eta, indexing='ij')  
        Sw = (Pi*Pj).flatten()
        
        Pxi, Pj = np.meshgrid(Px_xi, P_eta, indexing='ij')
        Swx = (Pxi*Pj*(2/a)).flatten()
        
        Pi, Pxj = np.meshgrid(P_xi, Px_eta, indexing='ij')
        Swy = (Pi*Pxj*(2/b)).flatten()

        Pxxi, Pj = np.meshgrid(Pxx_xi, P_eta, indexing='ij')
        Swxx = (Pxxi*Pj*(2/a)**2).flatten()

        Pi, Pxxj = np.meshgrid(P_xi, Pxx_eta, indexing='ij')
        Swyy = (Pi*Pxxj*(2/b)**2).flatten()

        Pxi, Pxj = np.meshgrid(Px_xi, Px_eta, indexing='ij')
        Swxy = (Pxi*Pxj*(2/a)*(2/b)).flatten()
        
        
        e1xx = -Swxx
        e1yy = -Swyy
        e1xy = -2*Swxy
        
        Mxx = prop.D11*e1xx + prop.D12*e1yy + prop.D16*e1xy
        Myy = prop.D12*e1xx + prop.D22*e1yy + prop.D26*e1xy
        Mxy = prop.D16*e1xx + prop.D26*e1yy + prop.D66*e1xy
        
        detJ = a*b/4
        
        # stiffness matrix
        addouter(K, detJ*weight*Mxx, e1xx)
        addouter(K, detJ*weight*Myy, e1yy)
        addouter(K, detJ*weight*Mxy, e1xy)
        

m1 = 20
m2 = 20
N = 400


#### External force vector

In [20]:
xi = 0 # x = a/2
eta = 0 # y = b/2
P_xi = vecf(m1, xi, xit1, xir1, xit2, xir2)
P_eta = vecf(m2, eta, etat1, etar1, etat2, etar2)
Pi, Pj = np.meshgrid(P_xi, P_eta, indexing='ij')  
Sw = (Pi*Pj).flatten()

Pforce = 1.

Fext = Pforce*Sw

#### Solve the linear system Ku = Fext for Ritz coefficients u

In [21]:
u = solve(csc_matrix(K), Fext)

			Removing null columns...
				144 columns removed
			finished!


#### Calculating displacement field for plotting

In [22]:
nx, ny = 40, 20
X, Y = np.meshgrid(
    np.linspace(0, a, nx),
    np.linspace(0, b, ny),
    indexing='ij'
)

W = np.zeros_like(X)

Su = np.zeros(N)
Sv = np.zeros(N)
Sw = np.zeros(N)

for ij, x in np.ndenumerate(X):
    xi = 2*x/a - 1
    eta = 2*Y[ij]/b - 1

    P_xi = vecf(m1, xi, xit1, xir1, xit2, xir2)
    P_eta = vecf(m2, eta, etat1, etar1, etat2, etar2)

    Pi, Pj = np.meshgrid(P_xi, P_eta, indexing='ij')
    Sw = (Pi*Pj).flatten()

    W[ij] = Sw @ u


#### Printing the maximum displacement

In [23]:
print('W.min(), W.max()', W.min(), W.max())
print('W.max() CLPT simply supported', 2.774985496231098e-05)

W.min(), W.max() -5.0936068922973684e-11 2.68373191220324e-05
W.max() CLPT simply supported 2.774985496231098e-05


In [46]:
import plotly.graph_objects as go

fig = go.Figure(data=[go.Surface(x=X, y=Y, z=W)])

fig.update_layout(
    title='Plate Deflection (CLPT)',
    scene=dict(
        xaxis_title='X (m)',
        yaxis_title='Y (m)',
        zaxis_title='W (m)',
        aspectmode='data',
        camera=dict(
            eye=dict(x=0, y=0, z=100*a/2),
            center=dict(x=a/2, y=b/2, z=0),
            up=dict(x=0, y=1, z=0)
        )
    ),
    width=800,
    height=600
)

fig.show()